In [2]:
import pandas as pd

X_train = pd.read_csv("X_train.csv")
X_val   = pd.read_csv("X_val.csv")

y_train = pd.read_csv("y_train.csv").squeeze()
y_val   = pd.read_csv("y_val.csv").squeeze()


In [4]:
from sklearn.preprocessing import LabelEncoder

cat_cols = X_train.select_dtypes(include="object").columns

for col in cat_cols:
    le = LabelEncoder()
    le.fit(X_train[col].astype(str))

    known = set(le.classes_)
    X_val_col = X_val[col].astype(str)
    X_val_col = X_val_col.where(X_val_col.isin(known), other="Unknown")

    # 确保 Unknown 在 classes 里
    if "Unknown" not in known:
        le.fit(list(le.classes_) + ["Unknown"])

    X_train[col] = le.transform(X_train[col].astype(str).where(
        X_train[col].astype(str).isin(set(le.classes_)), other="Unknown"
    ))
    X_val[col] = le.transform(X_val_col)


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)
rf.fit(X_train, y_train)

pred = rf.predict(X_val)
print(classification_report(y_val, pred))


              precision    recall  f1-score   support

           0       0.61      0.86      0.71      9871
           1       0.50      0.34      0.40      6398
           2       0.43      0.01      0.03      2049

    accuracy                           0.58     18318
   macro avg       0.51      0.40      0.38     18318
weighted avg       0.55      0.58      0.53     18318

